# 03 — Exploratory Data Analysis

Sanity-check all 20 features before modeling. Verify no lookahead, inspect distributions, correlations, and regime behavior.

In [ ]:
import sys, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sys.path.insert(0, os.path.abspath('..'))

from src.data.preprocess import FEATURE_COLS

PROCESSED_DIR = os.path.join('..', 'data', 'processed')
FIGURES_DIR = os.path.join('..', 'results', 'figures')
os.makedirs(FIGURES_DIR, exist_ok=True)

plt.rcParams['figure.figsize'] = (14, 6)
plt.rcParams['font.size'] = 11

In [ ]:
master = pd.read_csv(os.path.join(PROCESSED_DIR, 'master_df.csv'), index_col=0, parse_dates=True)
print(f'Master: {master.shape}')
master[FEATURE_COLS].describe().round(4)

## 1. SPY Price and Volume

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
master['spy_close'].plot(ax=axes[0], title='SPY Close Price')
axes[0].axvline(pd.Timestamp('2020-01-01'), color='red', linestyle='--', label='Train/Test split')
axes[0].legend()
axes[0].set_ylabel('Price ($)')

master['log_return'].plot(ax=axes[1], title='SPY Log Returns', alpha=0.7)
axes[1].axvline(pd.Timestamp('2020-01-01'), color='red', linestyle='--')
axes[1].set_ylabel('Log Return')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_spy_price_returns.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2. Volatility Features

In [ ]:
vol_cols = ['rv_5d', 'rv_21d', 'rv_63d', 'gk_vol', 'vix']
fig, ax = plt.subplots(figsize=(14, 6))
for col in vol_cols:
    if col in master.columns:
        master[col].plot(ax=ax, alpha=0.7, label=col)
ax.axvline(pd.Timestamp('2020-01-01'), color='red', linestyle='--', label='Train/Test split')
ax.set_title('Volatility Features Over Time')
ax.set_ylabel('Annualized Volatility')
ax.legend(ncol=3)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_volatility.png'), dpi=150, bbox_inches='tight')
plt.show()

## 3. VIX Term Structure

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
master['vix'].plot(ax=axes[0], label='VIX', alpha=0.8)
axes[0].axhline(15, color='green', linestyle=':', alpha=0.5, label='Low/Med boundary')
axes[0].axhline(25, color='orange', linestyle=':', alpha=0.5, label='Med/High boundary')
axes[0].legend()
axes[0].set_title('VIX Level and Regime Boundaries')

if 'vix_slope' in master.columns:
    master['vix_slope'].plot(ax=axes[1], label='VIX slope (3M-9D)', alpha=0.8)
if 'vix_6m_slope' in master.columns:
    master['vix_6m_slope'].plot(ax=axes[1], label='VIX 6M slope (6M-3M)', alpha=0.8)
axes[1].axhline(0, color='gray', linestyle='--')
axes[1].legend()
axes[1].set_title('VIX Term Structure Slopes')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_vix_term_structure.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Volatility Regime Distribution

In [ ]:
train_mask = master.index < '2020-01-01'
test_mask = master.index >= '2020-01-01'

regime_names = {0: 'Low (VIX<15)', 1: 'Medium (15-25)', 2: 'High (VIX>25)'}

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, mask, title in [(axes[0], train_mask, 'Train (2010-2019)'), (axes[1], test_mask, 'Test (2020-2023)')]:
    counts = master.loc[mask, 'vol_regime'].value_counts().sort_index()
    labels = [regime_names.get(int(k), str(k)) for k in counts.index]
    ax.bar(labels, counts.values, color=['#2ecc71', '#f39c12', '#e74c3c'])
    ax.set_title(title)
    ax.set_ylabel('Trading Days')
    for i, v in enumerate(counts.values):
        ax.text(i, v + 10, str(v), ha='center')
plt.suptitle('Volatility Regime Distribution')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_regime_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Feature Correlation Matrix

In [ ]:
corr = master[FEATURE_COLS].corr()
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, ax=ax,
            square=True, linewidths=0.5, annot_kws={'size': 7})
ax.set_title('Feature Correlation Matrix (Full Period)')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_correlation.png'), dpi=150, bbox_inches='tight')
plt.show()

## 6. Log Return Distribution (fat tails check)

In [ ]:
from scipy import stats

returns = master['log_return'].dropna()
kurt = returns.kurtosis()
skew = returns.skew()

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(returns, bins=100, density=True, alpha=0.7, label=f'SPY (kurt={kurt:.2f}, skew={skew:.2f})')

# Overlay normal distribution
x = np.linspace(returns.min(), returns.max(), 200)
ax.plot(x, stats.norm.pdf(x, returns.mean(), returns.std()),
        'r--', linewidth=2, label='Normal (kurt=0)')
ax.set_title('SPY Log Return Distribution vs Normal')
ax.set_xlabel('Log Return')
ax.set_ylabel('Density')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'eda_return_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()

print(f'Excess kurtosis: {kurt:.2f} (normal = 0, i.e. raw kurtosis = 3)')
print(f'This confirms fat tails — Black-Scholes assumes normality.')

## 7. Options Data EDA

In [ ]:
opts_test_path = os.path.join(PROCESSED_DIR, 'opts_test.pkl')
opts_train_path = os.path.join(PROCESSED_DIR, 'opts_train.pkl')

if os.path.exists(opts_test_path):
    opts_test = pd.read_pickle(opts_test_path)
    opts_train = pd.read_pickle(opts_train_path)
    
    print(f'Train options: {len(opts_train)}')
    print(f'Test options:  {len(opts_test)}')
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Moneyness distribution
    opts_test['moneyness_input'].hist(ax=axes[0], bins=50, alpha=0.7)
    axes[0].set_title('Test Set: Moneyness Distribution')
    axes[0].axvline(0.97, color='r', linestyle='--', alpha=0.5)
    axes[0].axvline(1.03, color='r', linestyle='--', alpha=0.5)
    
    # Time to expiry
    (opts_test['time_to_expiry'] * 365).hist(ax=axes[1], bins=50, alpha=0.7)
    axes[1].set_title('Test Set: Days to Expiry')
    axes[1].axvline(30, color='r', linestyle='--', alpha=0.5)
    axes[1].axvline(60, color='r', linestyle='--', alpha=0.5)
    
    # Mid price
    opts_test['mid_price'].hist(ax=axes[2], bins=50, alpha=0.7)
    axes[2].set_title('Test Set: Mid Price Distribution')
    
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'eda_options.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Options data not preprocessed yet. Run 02_preprocess first with options CSV.')

## 8. Lookahead Sanity Check

Verify that features at time t do not use data from time t.

In [ ]:
# rv_21d at day t should use returns from t-21 through t-1 (shifted before rolling).
# Quick verification: rv_21d should NOT have perfect correlation with today's return.

corr_rv_vs_return = master['rv_21d'].corr(master['log_return'])
corr_rv_vs_prev_return = master['rv_21d'].corr(master['log_return'].shift(1))

print(f'Correlation of rv_21d with today\'s return:      {corr_rv_vs_return:.4f}')
print(f'Correlation of rv_21d with yesterday\'s return:  {corr_rv_vs_prev_return:.4f}')
print()
print('If these are similar magnitude, no lookahead in rv_21d.')
print('If rv_21d had perfect correlation with today\'s return, there would be lookahead.')